# 06b · CircuitKIT cross-check of the refusal circuit (toolkit chaining)

Runs Lexsi Labs' **CircuitKIT** (EAP-IG, node level) on the matched calibration pairs and compares its ranking with the components our own attribution + exact patching found in 06. Uses the tested adapter `safetypulse.adapters.circuitkit` (attach the `safetypulse-pkg` dataset).

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · `safetypulse-pkg` · outputs of **03 matched**, **04**, **06 matched** · `sp-models` |
| **Reads** | `pairs.jsonl`, `token_sets.json`, `probe_bundle/` |
| **Writes** (`/kaggle/working/`) | `circuitkit/crosscheck.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cu130',
 'transformers': '5.20.2',
 'cuda_available': True,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2',
 'gpus': ['Tesla T4'],
 'capability': [7, 5],
 'vram_gb': 15.6}

In [2]:
import json, subprocess, sys
import torch

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "circuitkit==0.2.0"], check=True)
# Kaggle unwraps a dataset's single top-level folder, so find the package by its files and copy it into a
# directory actually named `safetypulse` before importing it.
import shutil
src = next(p.parent for p in sp.find_inputs("cli.py")
           if (p.parent / "check.py").exists() and (p.parent / "adapters").is_dir())
shutil.copytree(src, "/tmp/sp_pkg/safetypulse", dirs_exist_ok=True)
sys.path.insert(0, "/tmp/sp_pkg")
from safetypulse import ProbeBundle
from safetypulse.adapters import circuitkit as ck

MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
N_PAIRS = int(os.environ.get("SP_N_PAIRS", 64)) if not sp.SMOKE else 4   # EAP-IG is memory-heavy on a T4
K = 20
OUT = sp.out_dir("circuitkit")
pairs = [p for p in sp.load_jsonl(sp.find_input("pairs.jsonl")) if p["split"] == "calib"][:N_PAIRS]
token_sets = sp.load_json(sp.find_input("token_sets.json"))[MODEL]
bundle = ProbeBundle.load(sp.find_input("probe_bundle.json").parent)
print(len(pairs), "calib pairs;", len(bundle.components), "components in our bundle; layer", bundle.layer)

64 calib pairs; 20 components in our bundle; layer 27


In [3]:
with sp.gpu_timer() as t:
    result = ck.crosscheck(sp.model_source(MODEL), pairs, token_sets, bundle, k=K, dtype="float16", batch_size=2)
result["gpu_seconds"] = t["seconds"]
sp.save_json(result, OUT / "crosscheck.json")
{k: v for k, v in result.items() if not isinstance(v, (list, dict))}

copied CohereLabs__tiny-aya-global to local disk in 0.4 min


/usr/local/lib/python3.13/dist-packages/circuitkit/utils/device.py:49: FutureWarning: torch.cuda._set_allocator_settings is deprecated. Use torch._C._accelerator_setAllocatorSettings instead.
  torch.cuda.memory._set_allocator_settings("expandable_segments:True")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model /tmp/sp_models/CohereLabs__tiny-aya-global into HookedTransformer


11:44:31 • ==================================================


11:44:31 • Starting: Circuit Discovery


... (12 outputs left out)

11:44:41 ✗ Operation failed: Circuit Discovery


11:44:41 ✗ Error: CUDA out of memory. Tried to allocate 6.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 5.00 GiB is free. Including non-PyTorch memory, this process has 9.56 GiB memory in use. Of the allocated memory 9.40 GiB is allocated by PyTorch, and 35.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


11:44:41 ✗ Error in discover_circuit: CUDA out of memory. Tried to allocate 6.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 5.00 GiB is free. Including non-PyTorch memory, this process has 9.56 GiB memory in use. Of the allocated memory 9.40 GiB is allocated by PyTorch, and 35.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


CircuitKitError: CUDA out of memory. Tried to allocate 6.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 5.00 GiB is free. Including non-PyTorch memory, this process has 9.56 GiB memory in use. Of the allocated memory 9.40 GiB is allocated by PyTorch, and 35.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf) (Context: operation=discover_circuit, function=discover_circuit, args=({'model': {'name': '/tmp/sp_models/CohereLabs__tiny-aya-global', 'precision': 'float16'}, 'discovery': {'algorithm': 'eap-ig', 'task': 'safetypulse_refusal_a662e788', 'level': 'node', 'batch_size': 2, kwargs={'_model': HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint(name='hook_embed')
  (blocks): TypedModuleList(
    